# 🌌 Conway's Game of Life: High-Performance Computing (CPU vs CUDA)

Welcome to the **Conway's Game of Life** HPC project! The goal of this project is to explore and analyze **how GPU acceleration via CUDA outperforms standard CPU computing** on 2D cellular automata.

### 📌 Key Features & Architecture:
- **Modular Codebase**: Separation into `src.config`, `src.patterns`, `src.cpu_engine`, `src.cuda_engine`, `src.preview`, and `src.gui`.
- **Interactive Settings & Live Preview**: Configure Grid Size, Iterations, Patterns, Random Density, Seed, Edge Behavior, and Block Size with real-time visual feedback.
- **High Performance CUDA Engine**: 2D grid/thread blocks with double-buffered GPU memory.
- **Fair Benchmarking Harness**: Seed-synchronized initial states to strictly verify output correctness and measure pure kernel vs total speedup.

In [ ]:
# Step 1: System Hardware & Environment Verification
import sys
import numpy as np
import numba
from numba import cuda

print(f"Python Version: {sys.version.split()[0]}")
print(f"NumPy Version:  {np.__version__}")
print(f"Numba Version:  {numba.__version__}")

cuda_ready = False
try:
    cuda_ready = cuda.is_available()
    if cuda_ready:
        print(f"✅ CUDA GPU Detected: {[d.name for d in cuda.gpus]}")
    else:
        print("ℹ️ CUDA GPU driver detected, loading toolkit...")
except Exception as e:
    print(f"CUDA Status: {e}")

## ⚙️ Interactive Settings & Live Preview

The snippet below renders the **Settings Panel** at the top and the **Live Preview** directly underneath.
- Change the **Initial Pattern**, **Grid Size**, **Random Density**, or **Random Seed** to see the preview update immediately.
- Click the **Run Simulation / Benchmark** button to test the configuration.

In [ ]:
# Step 2: Render the In-Notebook Interactive Dashboard
from src.gui.ipy_widget import create_notebook_dashboard

dashboard = create_notebook_dashboard()

### 🖥️ (Optional) Launch Dedicated Desktop GUI Window
If you prefer a native OS window with the top settings panel, real-time canvas preview, and the Run button, run the cell below:

In [ ]:
# Step 3: Launch Native Tkinter Window (Optional)
from src.gui.tk_window import open_settings_window

# Uncomment and execute to pop up the desktop window:
# open_settings_window()

## 🧪 Programmatic Testing: Patterns & Engines
Let's test both the CPU and CUDA engines directly from Python code and visualize the evolution.

In [ ]:
# Step 4: Compare Patterns & Generations
import matplotlib.pyplot as plt
from src.config import SimulationConfig, PatternType, EdgeBehavior
from src.patterns import create_initial_grid
from src.cpu_engine import CPUEngine

# Create a Glider Gun pattern on a 64x64 grid
config = SimulationConfig(grid_size=64, pattern=PatternType.GLIDER_GUN.value, iterations=60)
init_grid = create_initial_grid(config)

# Advance 60 generations using CPUEngine
engine = CPUEngine(edge_behavior=EdgeBehavior.WRAP_AROUND.value)
final_grid, time_taken = engine.run(init_grid, config.iterations)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(init_grid, cmap='magma')
axes[0].set_title(f"Initial State ({config.pattern})")
axes[0].axis('off')

axes[1].imshow(final_grid, cmap='magma')
axes[1].set_title(f"After {config.iterations} Generations ({time_taken*1000:.1f} ms)")
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 📊 HPC Benchmark: CPU vs CUDA Scaling Experiment
Here we benchmark across multiple grid sizes (64, 128, 256, 512, 1024) to analyze how GPU acceleration scales with workload size.

In [ ]:
# Step 5: Benchmark Execution across Grid Sizes
from src.config import SimulationConfig, AVAILABLE_GRID_SIZES, RunMode
from src.benchmark import run_benchmark

benchmark_results = []
test_sizes = [64, 128, 256, 512]

print("Running HPC Scaling Benchmark...")
for size in test_sizes:
    cfg = SimulationConfig(
        grid_size=size,
        iterations=100,
        pattern="Random",
        random_density=0.25,
        random_seed=42,
        run_mode=RunMode.BOTH.value,
        cuda_block_size=(16, 16)
    )
    res = run_benchmark(cfg)
    benchmark_results.append(res)
    print(res.summary_table())
    print()


## 📈 Performance Comparison Plots (Assignment Report Visuals)
Visualize CPU vs CUDA execution times and the resulting speedup factor.

In [ ]:
# Step 6: Generate Comparison Graphs
sizes = [r.grid_size for r in benchmark_results if r.cpu_time_sec is not None]
cpu_times_ms = [r.cpu_time_sec * 1000 for r in benchmark_results if r.cpu_time_sec is not None]
gpu_kernel_times_ms = [r.cuda_kernel_time_sec * 1000 if r.cuda_kernel_time_sec else None for r in benchmark_results]

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(sizes, cpu_times_ms, 'o-', color='#ef4444', label='CPU (Vectorized NumPy)', linewidth=2)
if any(gpu_kernel_times_ms):
    valid_gpu = [t for t in gpu_kernel_times_ms if t is not None]
    plt.plot(sizes[:len(valid_gpu)], valid_gpu, 's-', color='#10b981', label='CUDA GPU (Kernel)', linewidth=2)
plt.xlabel('Grid Dimension (N x N)')
plt.ylabel('Execution Time (ms)')
plt.title('Computation Time vs Grid Size')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.subplot(1, 2, 2)
speedups = [r.kernel_speedup for r in benchmark_results if r.kernel_speedup is not None]
if speedups:
    plt.plot(sizes[:len(speedups)], speedups, '^-', color='#38bdf8', linewidth=2)
    plt.axhline(1.0, color='gray', linestyle='--', label='1x (Baseline)')
    plt.xlabel('Grid Dimension (N x N)')
    plt.ylabel('Speedup Factor (x)')
    plt.title('GPU Speedup Factor over CPU')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
else:
    plt.text(0.5, 0.5, 'Run GPU Benchmark\nto View Speedup', horizontalalignment='center')

plt.tight_layout()
plt.show()